### theprotocol.it - scrape all records
### WORKING

In [1]:
import httpx

def get_protocol_csrf_token():
    """Get XSRF-TOKEN cookie from /csrf-token endpoint."""
    url = "https://apus-api.theprotocol.it/csrf-token"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
    }
    with httpx.Client() as client:
        client.get(url, headers=headers)
        # print(client.cookies)
        return client.cookies
    
def fetch_protocol_offers(page_number=80, page_size=50):
    """Fetch job offers from theprotocol.it API for a given page."""
    url = f"https://apus-api.theprotocol.it/offers/_search?pageNumber={page_number}&orderby.field=Relevance&pageSize={page_size}"
    cookies = get_protocol_csrf_token()
    xsrf_token = cookies.get("XSRF-TOKEN")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Content-Type": "application/json",
        "x-xsrf-token": xsrf_token
    }
    with httpx.Client(cookies=cookies) as client:
        response = client.post(url, headers=headers)

        if not response.text:
            return None
        return response.json()['offers']

offers = fetch_protocol_offers()
print(len(offers))
# import json
# offers_json = json.dumps(offers, indent=4, ensure_ascii=False)
# print(offers_json)

50


### rocketjobs.pl - scrape all records

In [2]:
import httpx
def fetch_justjoin_offers():
    """Fetch job offers from justjoin.it API."""
    url = "https://api.rocketjobs.pl/v2/user-panel/offers/by-cursor"
    params = {
        # "categories[]": 19,
        "currency": "pln",
        "from": 000,
        "itemsCount": 100,
        # "keywords[]": "python",
        "orderBy": "DESC",
        # "remoteWorkOptions[]": "remote",
        "sortBy": "published"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7"
    }
    response = httpx.get(url, params=params, headers=headers)
    offers = response.json()['data']
    return offers
offers = fetch_justjoin_offers()
print(f"Fetched {len(offers)} offers from rocketjobs.pl")
# print(offers)
# for offer in offers:
#     print(offer)

Fetched 100 offers from rocketjobs.pl


### justjoin.it - Scrape all records

In [3]:
import httpx
def fetch_justjoin_offers():
    """Fetch job offers from justjoin.it API."""
    url = "https://api.justjoin.it/v2/user-panel/offers/by-cursor"
    params = {
        # "categories[]": 19,
        "currency": "pln",
        "from": 3300,
        "itemsCount": 100,
        # "keywords[]": "python",
        "orderBy": "DESC",
        "remoteWorkOptions[]": "remote",
        "sortBy": "published"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7"
    }
    response = httpx.get(url, params=params, headers=headers)
    offers = response.json()['data']
    return offers
offers = fetch_justjoin_offers()
print(f"Fetched {len(offers)} offers from justjoin.it")
# print(offers)
# for offer in offers:
#     print(offer)

Fetched 19 offers from justjoin.it


#### Solidjobs - scrape all records - filter from url
Poprawnie filtruje wszystko

In [4]:
# test_link_all_params = "https://solid.jobs/offers/it;cities=Krak%C3%B3w,Praca%20zdalna,Warszawa;categories=Programista;experiences=Regular,Junior;minimumSalary=3000;subcategories=Python"
# test_link_all_params = "https://solid.jobs/offers/it;cities=Krak%C3%B3w,Praca%20zdalna,Warszawa;categories=Tester;experiences=Regular,Junior;minimumSalary=3000;subcategories=In%C5%BCynier%20test%C3%B3w%20automatycznych"
test_link_all_params = "https://solid.jobs/offers/it;experiences=Senior;cities=Tr%C3%B3jmiasto;categories=Analityk;minimumSalary=25500"
import urllib.parse
import httpx
def parse_url_params(url):
    """Parse filter parameters from URL."""
    params = {}
    parts = url.split(';')
    for part in parts:
        if '=' in part:
            key, value = part.split('=', 1)
            decoded_value = urllib.parse.unquote(value)
            if key == "cities" and "Trójmiasto" in decoded_value:
                decoded_value = decoded_value.replace("Trójmiasto", "Gdańsk,Gdynia,Sopot")
            params[key] = decoded_value
    print(params)
    return params

def filter_offers(offers, cities=None, categories=None, subcategories=None, experiences=None, minimumSalary=None):
    """Filter job offers by parameters."""
    filtered = []
    for offer in offers:
        if cities:
            city_list = cities.split(',')
            remote_values = ["W całości", "Możliwa w całości"]
            is_remote = offer.get("remotePossible") in remote_values
            city = offer.get("companyCity") in city_list
            
            is_fully_remote = "Praca zdalna" in city_list and is_remote
            invalid_location = not(is_fully_remote or city)
            if invalid_location:
                continue # skip record
        if categories and offer.get("mainCategory") != categories:
            continue # skip record
        if subcategories and offer.get("subCategory") != subcategories:
            continue # skip record
        if experiences:
            exp_list = experiences.split(',')
            if offer.get("experienceLevel") not in exp_list:
                continue # skip record
        salary = offer.get("salaryRange", {})

        is_salary_below_minimum = minimumSalary and (salary["upperBound"] < float(minimumSalary))
        if is_salary_below_minimum:
            continue # skip record
        filtered.append(offer)
    return filtered

def get_all_offers():
    solidjobs = 'https://solid.jobs/api/offers?division=it&sortOrder=default'
    headers = {
        "Accept": "application/vnd.solidjobs.jobofferlist+json, application/json, text/plain, */*",
    }
    solid = httpx.get(solidjobs, headers=headers)
    return solid.json()


params = parse_url_params(test_link_all_params)
offers = get_all_offers()
filtered_offers = filter_offers(offers, **params)
# import json
# print(json.dumps(filtered_offers, ensure_ascii=False, indent=4))
print(len(filtered_offers))
for offer in filtered_offers:
    print(offer['id'], offer['jobTitle'], offer['companyName'], offer['companyCity'], offer['experienceLevel'])

{'experiences': 'Senior', 'cities': 'Gdańsk,Gdynia,Sopot', 'categories': 'Analityk', 'minimumSalary': '25500'}
1
24017 Expert IT Analyst (Payments) emagine Gdańsk Senior


# NON-SELENIUM SITES - GET ALL RECORDS

In [ ]:
BULLDOGJOB = "https://bulldogjob.pl"
INHIRE = "https://inhire.io"

### NoFluffJobs.com - working

In [22]:
def fetch_nofluffjobs_offers(page=1, page_size=5000):
    """Fetch job offers from NoFluffJobs API."""
    url = "https://nofluffjobs.com/api/joboffers/main"
    params = {
        "pageTo": page,
        "pageSize": page_size,
        # "withSalaryMatch": "true",
        "salaryCurrency": "PLN",
        "salaryPeriod": "month",
        # "region": "pl",
        # "language": "pl-PL"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pl-PL,pl;q=0.9"
    }
    response = httpx.get(url, params=params, headers=headers)
    return response.json()

offers = fetch_nofluffjobs_offers()
# print(f"Fetched {len(offers)} offers from NoFluffJobs")
print(offers['totalCount'])
print(len(offers['postings']))

21420
21420


### PRACUJ.PL

### bulldogjob.pl

In [27]:
import httpx

def fetch_bulldogjob_offers():
    """Fetch all job offers from bulldogjob.pl GraphQL API."""
    url = "https://bulldogjob.pl/graphql"
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "*/*",
    }
    payload = {
        "operationName": "jobOffers",
        "variables": {
            "amount": 100,
            "language": "pl",
            "country": "PL"
        },
        "query": "query jobOffers($amount: Int, $language: LocaleEnum!, $country: String) {\n  jobOffers(amount: $amount, language: $language, country: $country) {\n    id\n    title\n    company { name }\n    city\n    salaryFrom\n    salaryTo\n    experienceLevel\n    remote\n    publishedAt\n    technologies { name }\n    employmentType\n    skills { name }\n    description\n    applyUrl\n  }\n}\n"
    }
    response = httpx.post(url, headers=headers, json=payload)
    return response.json()

offers = fetch_bulldogjob_offers()
print(offers)

{'errors': [{'message': "Field 'jobOffers' doesn't exist on type 'Query'", 'locations': [{'line': 2, 'column': 3}], 'path': ['query jobOffers', 'jobOffers'], 'extensions': {'code': 'undefinedField', 'typeName': 'Query', 'fieldName': 'jobOffers'}}, {'message': 'Variable $amount is declared by jobOffers but not used', 'locations': [{'line': 1, 'column': 1}], 'path': ['query jobOffers'], 'extensions': {'code': 'variableNotUsed', 'variableName': 'amount'}}, {'message': 'Variable $language is declared by jobOffers but not used', 'locations': [{'line': 1, 'column': 1}], 'path': ['query jobOffers'], 'extensions': {'code': 'variableNotUsed', 'variableName': 'language'}}, {'message': 'Variable $country is declared by jobOffers but not used', 'locations': [{'line': 1, 'column': 1}], 'path': ['query jobOffers'], 'extensions': {'code': 'variableNotUsed', 'variableName': 'country'}}]}
